# BLANC Tutorial: Unified API for Knowledge Base Reasoning

**B**uilding **L**ogical **A**bductive **N**on-monotonic **C**orpora

This notebook provides a comprehensive introduction to the BLANC framework for querying heterogeneous knowledge bases to support research on abductive and defeasible reasoning in foundation models.

## What You'll Learn

1. Theory construction and manipulation
2. Query building (deductive, abductive, defeasible)
3. Working with Prolog and ASP backends
4. Loading and querying knowledge bases
5. Research applications for LLM evaluation

## Prerequisites

- Python 3.11+
- BLANC installed: `pip install -e ".[dev]"`
- Optional: SWI-Prolog for Prolog backend

## 1. Setup and Imports

In [ ]:
# Core imports
from blanc import KnowledgeBase, Query
from blanc.core.theory import Rule, RuleType, Theory
from blanc.core.result import Result, ResultSet

# Utility imports
from pathlib import Path
import sys

# Check backend availability
from blanc.backends.asp import CLINGO_AVAILABLE
from blanc.backends.prolog import PYSWIP_AVAILABLE

print(f"ASP (Clingo) Backend: {'✓ Available' if CLINGO_AVAILABLE else '✗ Not Available'}")
print(f"Prolog (PySwip) Backend: {'✓ Available' if PYSWIP_AVAILABLE else '✗ Not Available'}")
print(f"\nPython version: {sys.version}")

## 2. Theory Construction

The core of BLANC is the `Theory` class, which represents knowledge bases with support for multiple rule types.

### 2.1 Simple Theory with Facts and Rules

In [ ]:
# Create a theory about mortality
theory = Theory()

# Add facts
theory.add_fact("human(socrates)")
theory.add_fact("human(plato)")
theory.add_fact("human(aristotle)")

# Add a strict rule (classical implication)
mortality_rule = Rule(
    head="mortal(X)",
    body=("human(X)",),
    rule_type=RuleType.STRICT
)
theory.add_rule(mortality_rule)

print("Theory in Prolog format:")
print(theory.to_prolog())
print("\nTheory in defeasible logic format:")
print(theory.to_defeasible())

### 2.2 Rule Types: The Power of Defeasible Logic

BLANC supports four types of rules:

1. **FACT**: Ground truths
2. **STRICT**: Classical implications (`→`) - always hold
3. **DEFEASIBLE**: Defeasible rules (`⇒`) - typically hold but can be defeated
4. **DEFEATER**: Defeaters (`⇝`) - block inferences without asserting opposites

In [ ]:
# Classic Tweety example
tweety_theory = Theory()

# Facts
tweety_theory.add_fact("bird(tweety)")
tweety_theory.add_fact("bird(woody)")
tweety_theory.add_fact("penguin(opus)")

# Defeasible rule: birds typically fly
r1 = Rule(
    head="flies(X)",
    body=("bird(X)",),
    rule_type=RuleType.DEFEASIBLE,
    label="r1"
)
tweety_theory.add_rule(r1)

# Strict rule: penguins are birds
r2 = Rule(
    head="bird(X)",
    body=("penguin(X)",),
    rule_type=RuleType.STRICT
)
tweety_theory.add_rule(r2)

# Defeasible rule: penguins typically don't fly
r3 = Rule(
    head="not_flies(X)",
    body=("penguin(X)",),
    rule_type=RuleType.DEFEASIBLE,
    label="r3"
)
tweety_theory.add_rule(r3)

# Superiority: penguin rule defeats bird rule
tweety_theory.add_superiority("r3", "r1")

print("Tweety Theory (Defeasible Logic Format):")
print(tweety_theory.to_defeasible())
print(f"\nTotal rules: {len(tweety_theory.rules)}")
print(f"Total facts: {len(tweety_theory.facts)}")

## 3. Query Building

BLANC provides a fluent query interface supporting three reasoning modes.

### 3.1 Deductive Queries

Standard logical inference: find all solutions that satisfy the query.

In [ ]:
# Example query construction (execution requires backend)
# This shows the API syntax

# Simple selection
query1 = Query(None).select("mortal(X)")
print("Query 1 (Simple selection):")
print(f"  Type: {query1.query_type}")
print(f"  Goal: {query1.goal}")

# Selection with conditions
query2 = Query(None).select("diagnosis(Patient, Disease)").where("symptom(Patient, fever)", "symptom(Patient, cough)")
print("\nQuery 2 (With conditions):")
print(f"  Type: {query2.query_type}")
print(f"  Goal: {query2.goal}")
print(f"  Conditions: {query2.conditions}")

# With result limit
query3 = Query(None).select("person(X)").limit(5)
print("\nQuery 3 (With limit):")
print(f"  Goal: {query3.goal}")
print(f"  Limit: {query3.result_limit}")

### 3.2 Abductive Queries

Hypothesis generation: find explanations for observations.

In [ ]:
# Abductive query for medical diagnosis
abd_query = (Query(None)
    .abduce("infected(john, covid)")
    .given("symptom(john, fever)", "symptom(john, cough)")
    .with_hypotheses("recent_travel(john)", "contact_with_infected(john)", "immunocompromised(john)")
    .minimize("hypothesis_count")
)

print("Abductive Query:")
print(f"  Type: {abd_query.query_type}")
print(f"  Observation: {abd_query.goal}")
print(f"  Evidence: {abd_query.conditions}")
print(f"  Candidate Hypotheses: {abd_query.hypotheses}")
print(f"  Minimize: {abd_query.minimize_criterion}")

### 3.3 Defeasible Queries

Non-monotonic reasoning: what can be derived given potential defeaters?

In [ ]:
# Defeasible query: can we infer Tweety flies?
def_query = (Query(None)
    .defeasibly_infer("flies(tweety)")
    .assuming("bird(tweety)")
    .with_defeaters("wounded(tweety)")  # What if Tweety is wounded?
)

print("Defeasible Query:")
print(f"  Type: {def_query.query_type}")
print(f"  Goal: {def_query.goal}")
print(f"  Assumptions: {def_query.defeasible_context.assumptions}")
print(f"  Defeaters: {def_query.defeasible_context.defeaters}")

## 4. Working with ASP Backend (Clingo)

The ASP backend uses Answer Set Programming for powerful reasoning capabilities.

In [ ]:
if CLINGO_AVAILABLE:
    # Load theory with ASP backend
    kb_asp = KnowledgeBase(backend='asp')
    kb_asp.load(theory)
    
    print("✓ ASP Knowledge Base loaded successfully")
    print(f"Backend: {kb_asp.backend_name}")
    print(f"\nProgram:")
    print(kb_asp.backend._program_text)
else:
    print("⚠ Clingo not available. Install with: pip install clingo clorm")

### 4.1 ASP Deductive Query Example

In [ ]:
if CLINGO_AVAILABLE:
    # Create a simple ASP program
    simple_kb = KnowledgeBase(backend='asp')
    simple_program = """
    person(alice).
    person(bob).
    person(charlie).
    
    human(X) :- person(X).
    mortal(X) :- human(X).
    """
    simple_kb.load(simple_program)
    
    # Solve and display results
    with simple_kb.backend._control.solve(yield_=True) as handle:
        for i, model in enumerate(handle):
            atoms = [str(atom) for atom in model.symbols(shown=True)]
            print(f"Model {i+1}:")
            print(f"  Atoms: {sorted(atoms)}")
            break  # Just show first model
else:
    print("Skipping ASP example (Clingo not available)")

### 4.2 ASP Abductive Reasoning Example

In [ ]:
if CLINGO_AVAILABLE:
    # Abductive reasoning with choice rules
    abd_kb = KnowledgeBase(backend='asp')
    abd_program = """
    % Observation: alarm is sounding
    alarm.
    
    % Possible explanations (choice rules)
    { burglary }.
    { earthquake }.
    
    % Alarm sounds if either event occurs
    alarm :- burglary.
    alarm :- earthquake.
    
    % Minimize number of explanations (Occam's Razor)
    #minimize { 1 : burglary; 1 : earthquake }.
    """
    abd_kb.load(abd_program)
    
    print("Abductive Reasoning: Why is the alarm sounding?\n")
    with abd_kb.backend._control.solve(yield_=True) as handle:
        for i, model in enumerate(handle, 1):
            atoms = [str(atom) for atom in model.symbols(shown=True)]
            explanations = [a for a in atoms if a in ['burglary', 'earthquake']]
            print(f"Explanation {i}: {explanations if explanations else ['No specific cause needed']}")
            if i >= 3:  # Show first 3 models
                break
else:
    print("Skipping abductive example (Clingo not available)")

## 5. Working with Prolog Backend (SWI-Prolog)

The Prolog backend provides traditional logic programming capabilities with backtracking.

In [ ]:
if PYSWIP_AVAILABLE:
    # Load theory with Prolog backend
    kb_prolog = KnowledgeBase(backend='prolog')
    kb_prolog.load(theory)
    
    print("✓ Prolog Knowledge Base loaded successfully")
    print(f"Backend: {kb_prolog.backend_name}")
    
    # Query all mortals
    print("\nQuerying: mortal(X)")
    for solution in kb_prolog.backend._prolog.query("mortal(X)"):
        print(f"  X = {solution['X']}")
else:
    print("⚠ PySwip/SWI-Prolog not available.")
    print("Install SWI-Prolog: https://www.swi-prolog.org/download/stable")
    print("Then: pip install pyswip")

## 6. Loading Knowledge Bases from Files

BLANC can load knowledge bases from Prolog (.pl) and ASP (.lp) files.

### 6.1 Medical Diagnosis Example

In [ ]:
# Path to medical knowledge base
medical_kb_path = Path("../examples/knowledge_bases/medical.pl")

if medical_kb_path.exists():
    print(f"Loading: {medical_kb_path}")
    print("\nKnowledge Base Contents:")
    print(medical_kb_path.read_text()[:500] + "...\n")
    
    if PYSWIP_AVAILABLE:
        # Load with Prolog
        medical_kb = KnowledgeBase(backend='prolog')
        medical_kb.load(medical_kb_path)
        
        # Query diagnoses
        print("Querying diagnoses:")
        for solution in medical_kb.backend._prolog.query("diagnosis(P, D)"):
            print(f"  Patient {solution['P']} diagnosed with: {solution['D']}")
    else:
        print("(Prolog backend not available for query execution)")
else:
    print(f"Knowledge base not found at: {medical_kb_path}")

### 6.2 Family Relations Example

In [ ]:
family_kb_path = Path("../examples/knowledge_bases/family.pl")

if family_kb_path.exists() and PYSWIP_AVAILABLE:
    family_kb = KnowledgeBase(backend='prolog')
    family_kb.load(family_kb_path)
    
    print("Family Knowledge Base Loaded\n")
    
    # Find all grandparents
    print("Grandparent relationships:")
    for solution in family_kb.backend._prolog.query("grandparent(GP, GC)"):
        print(f"  {solution['GP']} is grandparent of {solution['GC']}")
    
    # Find ancestors
    print("\nAll of Tom's descendants:")
    for solution in family_kb.backend._prolog.query("descendant(D, tom)"):
        print(f"  {solution['D']}")
elif not family_kb_path.exists():
    print(f"Family KB not found at: {family_kb_path}")
else:
    print("Prolog backend required for family KB queries")

## 7. IDP Discovery Scenario (From Research Paper)

This example demonstrates how defeasible logic models transformational scientific discoveries.

In [ ]:
idp_kb_path = Path("../examples/knowledge_bases/idp_discovery.pl")

if idp_kb_path.exists():
    print("IDP (Intrinsically Disordered Proteins) Discovery Scenario\n")
    print("This demonstrates how defeasible logic can model paradigm shifts in science.\n")
    
    # Show the knowledge base
    content = idp_kb_path.read_text()
    print("Key Rules:")
    for line in content.split('\n'):
        if 'functional(P)' in line or 'intrinsically_disordered(P)' in line:
            print(f"  {line.strip()}")
    
    if PYSWIP_AVAILABLE:
        idp_kb = KnowledgeBase(backend='prolog')
        idp_kb.load(idp_kb_path)
        
        print("\nQueries:")
        
        # Check if alpha-synuclein is functional
        print("\n1. Is alpha-synuclein functional?")
        solutions = list(idp_kb.backend._prolog.query("functional(alpha_synuclein)"))
        print(f"   Result: {'Yes' if solutions else 'No'} (via IDP exception to structure-function dogma)")
        
        # Find all IDPs
        print("\n2. Which proteins are intrinsically disordered?")
        for solution in idp_kb.backend._prolog.query("intrinsically_disordered(P)"):
            print(f"   {solution['P']}")
        
        # Check paradigm shift
        print("\n3. Does this represent a paradigm shift?")
        solutions = list(idp_kb.backend._prolog.query("paradigm_shift(X)"))
        print(f"   Result: {'Yes' if solutions else 'No'}")
    else:
        print("\n(Prolog backend required for query execution)")
else:
    print(f"IDP knowledge base not found at: {idp_kb_path}")

## 8. Format Conversion

BLANC can convert theories between different formats.

In [ ]:
# Create a theory
conv_theory = Theory()
conv_theory.add_fact("bird(tweety)")
conv_theory.add_rule(
    Rule(
        head="flies(X)",
        body=("bird(X)",),
        rule_type=RuleType.DEFEASIBLE,
        label="r1"
    )
)

print("Original Theory")
print("=" * 50)

print("\n1. Prolog Format:")
print(conv_theory.to_prolog())

print("\n2. ASP Format:")
print(conv_theory.to_asp())

print("\n3. Defeasible Logic Format:")
print(conv_theory.to_defeasible())

## 9. Research Applications

### 9.1 Generating Incomplete Theories

For abductive reasoning research, we need incomplete theories that require hypothesis generation.

In [ ]:
# Create a complete theory
complete_theory = Theory()
complete_theory.add_fact("symptom(patient1, fever)")
complete_theory.add_fact("symptom(patient1, cough)")
complete_theory.add_fact("recent_travel(patient1)")
complete_theory.add_rule(
    Rule(
        head="diagnosis(P, covid)",
        body=("symptom(P, fever)", "symptom(P, cough)", "recent_travel(P)")
    )
)

print("Complete Theory:")
print(complete_theory.to_prolog())

# Create incomplete version (remove travel fact)
incomplete_theory = Theory()
incomplete_theory.add_fact("symptom(patient1, fever)")
incomplete_theory.add_fact("symptom(patient1, cough)")
# Missing: recent_travel(patient1) <- This must be abduced
incomplete_theory.add_rule(
    Rule(
        head="diagnosis(P, covid)",
        body=("symptom(P, fever)", "symptom(P, cough)", "recent_travel(P)")
    )
)

print("\nIncomplete Theory (for abductive task):")
print(incomplete_theory.to_prolog())
print("\nTask: Given observation 'diagnosis(patient1, covid)',")
print("      what hypothesis completes the theory?")
print("Answer: recent_travel(patient1) [gold hypothesis]")

### 9.2 Dataset Generation Pipeline

Creating evaluation instances for foundation models.

In [ ]:
# Example dataset entry structure
dataset_entry = {
    "id": "example_001",
    "domain": "medical",
    "incomplete_theory": incomplete_theory.to_prolog(),
    "observation": "diagnosis(patient1, covid)",
    "evidence": [
        "symptom(patient1, fever)",
        "symptom(patient1, cough)"
    ],
    "gold_hypotheses": ["recent_travel(patient1)"],
    "distractor_hypotheses": [
        "immunocompromised(patient1)",
        "contact_with_infected(patient1)"
    ],
    "difficulty": "medium",
    "reasoning_type": "abductive"
}

print("Dataset Entry Example:")
import json
print(json.dumps(dataset_entry, indent=2))

## 10. Performance and Benchmarking

In [ ]:
import time

if CLINGO_AVAILABLE:
    # Benchmark ASP backend
    print("Benchmarking ASP Backend\n")
    
    # Create test program
    test_program = """
    % Generate numbers 1-100
    number(1..100).
    
    % Find even numbers
    even(X) :- number(X), X \\ 2 = 0.
    
    % Find prime-like (simplified)
    prime_like(X) :- number(X), X > 1, X \\ 2 != 0, X \\ 3 != 0.
    """
    
    bench_kb = KnowledgeBase(backend='asp')
    
    # Time loading
    start = time.time()
    bench_kb.load(test_program)
    load_time = time.time() - start
    
    # Time solving
    start = time.time()
    with bench_kb.backend._control.solve(yield_=True) as handle:
        for model in handle:
            atom_count = len(list(model.symbols(shown=True)))
            break
    solve_time = time.time() - start
    
    print(f"Load time: {load_time*1000:.2f} ms")
    print(f"Solve time: {solve_time*1000:.2f} ms")
    print(f"Atoms in model: {atom_count}")
else:
    print("Clingo not available for benchmarking")

## 11. Summary and Next Steps

### What We Covered

1. ✅ Theory construction with multiple rule types
2. ✅ Query building (deductive, abductive, defeasible)
3. ✅ ASP backend with Clingo
4. ✅ Prolog backend with PySwip (if available)
5. ✅ Loading knowledge bases from files
6. ✅ Format conversion
7. ✅ Research applications (dataset generation)

### Next Steps

1. **Install SWI-Prolog** (if not already): https://www.swi-prolog.org/download/stable
2. **Explore more knowledge bases**: Check `examples/knowledge_bases/`
3. **Download historical KBs**: TaxKB, NephroDoctor (see Phase 2 plan)
4. **Create your own theories**: For your research domain
5. **Generate datasets**: Use incomplete theories for LLM evaluation
6. **Read the paper**: `paper/paper.tex` for theoretical background

### Resources

- Documentation: `Guidance_Documents/API_Design.md`
- Examples: `examples/basic_usage.py`
- Tests: `tests/` directory
- Installation: `INSTALL.md`

## 12. Exercises

Try these exercises to practice:

### Exercise 1: Create Your Own Theory

Create a theory about animals and their properties. Include:
- At least 5 facts
- At least 2 strict rules
- At least 1 defeasible rule

### Exercise 2: Abductive Reasoning

Given the incomplete medical theory above, create an abductive query that finds the missing hypothesis.

### Exercise 3: Load and Query

Load one of the example knowledge bases and write 3 different queries to explore it.

### Exercise 4: Format Conversion

Create a theory and convert it to all three formats (Prolog, ASP, Defeasible Logic). Compare the outputs.

---

**BLANC Framework**  
Building Logical Abductive Non-monotonic Corpora  
For questions and support, see the GitHub repository.

Last updated: February 5, 2026